In [ ]:
import pandas as pd

#laoding input file data manually 
df = pd.read_csv("/Users/rkafle/Desktop/Research_AI/clinical_genotype_HGB.csv")

# Table 1:Identifiers and Demographics
cols_table1 = ['wihsid', 'bsdate', 'bsvisit', 'dob', 'status']
table1 = df[cols_table1]

#convert numeric date columns to actual dates
base_date = pd.Timestamp('1960-01-01')
table1['bsdate'] = base_date + pd.to_timedelta(table1['bsdate'], unit='D')
table1['dob'] = base_date + pd.to_timedelta(table1['dob'], unit='D')


print(table1.head())
print(table1.info())


: 

In [7]:

# Remove duplicates and handle missing
table1 = table1.drop_duplicates().dropna(subset=['wihsid'])

# Convert status to categorical
status_map = {
    1: "Negative",
    2: "Prevalent",
    4: "Converter",
    5: "Converter_Death"
}
table1['status_label'] = table1['status'].map(status_map)



In [ ]:
# Example: derive approximate age at baseline
table1['age_baseline'] = (table1['bsdate'] - table1['dob']).dt.days / 365.25

# Basic statistics
print(table1.groupby('status_label')['age_baseline'].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data=table1, x='status_label')
plt.title("Distribution of Participant Health Status")
plt.show()

sns.boxplot(data=table1, x='status_label', y='age_baseline')
plt.title("Baseline Age by Health Status")
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = table1[['bsvisit', 'age_baseline']].fillna(0)
y = table1['status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))
